In [5]:
# ==========================================
# VALIDATION CHECK: Missing or Corrupted TIFs
# ==========================================
import os
import rasterio
import numpy as np
import ee

import shutil

# If using Colab, uncomment the next two lines to mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy files to local Colab storage before running the check
#shutil.copytree('/content/drive/MyDrive/Sentinel2_Training_Data', '/content/local_training_data')

# Initialize Earth Engine to fetch your master list of clusters
try:
    ee.Initialize(project='integrated-hawk-485001-k3')
except ee.EEException:
    ee.Authenticate()
    ee.Initialize(project='integrated-hawk-485001-k3')

# 1. Define where your tasks saved the files
# Update this path to match your actual Google Drive folder
drive_export_folder = '/content/drive/MyDrive/Sentinel2_Training_Data'

# 2. Load your raw DHS points asset to get the list of cluster IDs
ASSET_ID = 'projects/integrated-hawk-485001-k3/assets/PH_DHS_GPS'
dhs_points = ee.FeatureCollection(ASSET_ID)

# Pull the feature data
features_list = dhs_points.getInfo()['features']
total_clusters = len(features_list)
expected_quarters = [1, 2, 3, 4]

missing_files = []
invalid_black_files = []
valid_count = 0

print(f"Scanning {drive_export_folder} for {total_clusters * 4} expected images...")

for feature in features_list:
    cluster_id = str(feature['properties']['DHSCLUST'])
    print(f"Checking cluster {cluster_id}")

    for q_num in expected_quarters:
        filename = f"dhs_{cluster_id}_2022_Q{q_num}.tif"
        filepath = os.path.join(drive_export_folder, filename)

        # Check 1: Does the file even exist?
        if not os.path.exists(filepath):
            missing_files.append(filename)
            continue

        # Check 2: Is the file corrupted or 100% black (no data)?
        try:
            with rasterio.open(filepath) as src:
        #        # Read the first band (Red) to check for pixel data
                band_data = src.read(1)

                # If every single pixel in the array is 0, the image is dead/black
                if np.all(band_data == 0):
                    invalid_black_files.append(filename)
                    print(f"Black/Corrupted: {filename}")
                else:
                    valid_count += 1

        except rasterio.errors.RasterioIOError:
            # This catches files that are technically there but corrupted/unreadable
            invalid_black_files.append(f"{filename} (Corrupted)")

# Print the final diagnostic report
print("-" * 40)
print("VALIDATION REPORT")
print("-" * 40)
print(f"Total Expected:    {total_clusters * 4}")
print(f"Total Valid:       {valid_count}")
print(f"Missing Files:     {len(missing_files)}")
print(f"Black/Corrupted:   {len(invalid_black_files)}")

if missing_files or invalid_black_files:
    print("\nACTION REQUIRED: Please queue an Earth Engine export for the following files:")
    if missing_files:
        print(f"\nMissing ({len(missing_files)}):", missing_files[:10], "...(truncated)" if len(missing_files) > 10 else "")
    if invalid_black_files:
        print(f"\nInvalid/Black ({len(invalid_black_files)}):", invalid_black_files[:10], "...(truncated)" if len(invalid_black_files) > 10 else "")
else:
    print("\n✅ SUCCESS: All images are present and contain valid pixel data! Ready for VGG16 extraction.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Scanning /content/drive/MyDrive/Sentinel2_Training_Data for 4988 expected images...
Checking cluster 559
Checking cluster 1067
Checking cluster 1092
Checking cluster 1123
Checking cluster 1091
Checking cluster 294
Checking cluster 304
Checking cluster 307
Checking cluster 310
Checking cluster 358
Checking cluster 378
Checking cluster 400
Checking cluster 402
Checking cluster 403
Checking cluster 8
Checking cluster 61
Checking cluster 65
Checking cluster 413
Checking cluster 414
Checking cluster 415
Checking cluster 417
Checking cluster 418
Checking cluster 419
Checking cluster 421
Checking cluster 422
Checking cluster 423
Checking cluster 424
Checking cluster 425
Checking cluster 426
Checking cluster 427
Checking cluster 429
Checking cluster 430
Checking cluster 431
Checking cluster 432
Checking cluster 433
Checking cluster 434
Checking cluster 436
Checking c